# Importar Librerías


In [1]:
import pandas as pd
from pathlib import Path
import joblib, re, unicodedata

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [2]:
import sys

BASE_DIR = Path.cwd().parent

if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from oci_service import OCIService

# oci_service = OCIService(bucket_name="bucket", config_path=str(BASE_DIR / ".config"))

# Carga de `csv` a usar


In [3]:
# Dataset A
df_clasificacion_gastos = pd.read_csv("../data/processed/transacciones_procesadas.csv")
# Dataset B
df_perfil_financiero = pd.read_csv("../data/processed/perfiles_procesados.csv")

In [4]:
df_clasificacion_gastos

,descripcion,categoria,monto,descripcion_limpia
0,*supermercado*,Alimentación,647.41,supermercado
1,COMPRA EN SUPERMERCADO,Alimentación,498.99,compra en supermercado
2,*supermercado*,Alimentación,384.37,supermercado
3,*supermercado*,Alimentación,93.56,supermercado
4,COMPRA EN SUPERMERCADO,Alimentación,300.28,compra en supermercado
...,...,...,...,...
1705,DOMESTIKA,Educación,1995.49,domestika
1706,compra dermatologo,Salud,1445.62,compra dermatologo
1707,FARMACIAS DEL AHORRO,Salud,668.42,farmacias del ahorro
1708,Zoologico,Ocio y Servicios,270.42,zoologico


In [5]:
df_perfil_financiero

,ingreso_mensual,proporcion_gasto,gasto_total,nivel_endeudamiento,frecuencia_ahorro,perfil_financiero,ratio_gasto_ingreso,frecuencia_ahorro_encoded
0,64845.55,22.54,14615.80,37.45,Alta,Saludable,0.225394,2.0
1,8115.18,22.60,1834.03,95.07,Media,En Riesgo,0.226000,1.0
2,41719.30,8.85,3692.37,73.20,Baja,En Riesgo,0.088505,0.0
3,17860.93,82.32,14703.51,59.87,Media,En Observación,0.823222,1.0
4,9478.25,74.70,7080.20,15.60,Media,En Observación,0.746994,1.0
...,...,...,...,...,...,...,...,...
695,99808.24,88.25,88080.48,61.06,Media,En Observación,0.882497,1.0
696,52844.73,65.65,34691.14,28.86,Baja,En Observación,0.656473,0.0
697,31348.28,71.63,22455.14,58.12,Media,En Observación,0.716312,1.0
698,74823.11,88.78,66428.69,15.44,Alta,En Observación,0.887810,2.0


# EDA Y Procesamiento De Notebooks

Para mas información sobre los EDA y procesamientos de los dataset, consultar los notebooks dedicados a cada uno de ellos, por favor.

- _G9-LATAM-Team-35-FinanceAI\data-science\notebooks\01_yul_clasificacion_gastos.ipynb_
- _G9-LATAM-Team-35-FinanceAI\data-science\notebooks\02_marco_perfil_financiero.ipynb_


# Entrenamiento de modelos


### Dataset Clasificación De Gastos (A)


In [6]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = (
        unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")
    )
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


stop_words_es = [
    "de",
    "la",
    "que",
    "el",
    "en",
    "y",
    "a",
    "los",
    "del",
    "se",
    "las",
    "por",
    "un",
    "para",
    "con",
    "no",
    "una",
    "su",
    "al",
    "lo",
    "como",
    "pero",
    "sus",
    "o",
    "este",
    "esta",
    "entre",
    "cuando",
    "sin",
    "sobre",
    "tambien",
    "me",
    "hasta",
    "hay",
    "donde",
    "desde",
    "todo",
    "nos",
    "durante",
    "todos",
    "uno",
    "les",
    "ni",
    "contra",
    "otros",
    "ese",
    "eso",
    "ante",
    "ellos",
    "esto",
]

df_clasificacion_gastos["descripcion_limpia"] = df_clasificacion_gastos[
    "descripcion"
].apply(limpiar_texto)

X = df_clasificacion_gastos["descripcion_limpia"]
y = df_clasificacion_gastos["categoria"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline_gastos = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                max_features=600, stop_words=stop_words_es, ngram_range=(1, 2)
            ),
        ),
        ("classifier", LogisticRegression(random_state=42)),
    ]
)

pipeline_gastos.fit(X_train, y_train)

output_dir = Path("../models")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "clasificador_gastos.joblib"

joblib.dump(pipeline_gastos, output_path, compress=3)

# oci_service.subir_archivo(
#     nombre_objeto="clasificador_gastos.joblib",  # Nombre en OCI (destino)
#     ruta_local=output_path                             # Ruta local (origen)
# )

['../models/clasificador_gastos.joblib']

Se uso **Regresión Logística** según la interpretación de datos en _..\notebooks\03_luz_bi_analysis.ipynb_

> Tras comparar los modelos candidatos mediante validación cruzada, la **Regresión Logística** fue seleccionada como el algoritmo con mejor desempeño para la clasificación automática de gastos. Posteriormente, el modelo fue evaluado utilizando el conjunto de prueba, obteniendo un **Accuracy del 71%**, lo que indica que aproximadamente siete de cada diez transacciones fueron clasificadas correctamente.
>
> Las métricas de **Precision**, **Recall** y **F1-Score** muestran un comportamiento equilibrado entre las distintas categorías, aunque con diferencias derivadas de la complejidad del problema. La categoría **Vivienda** presentó el mejor desempeño, alcanzando un F1-Score de **0.87**, lo que sugiere que las descripciones asociadas a este tipo de gasto contienen términos distintivos que facilitan su identificación por parte del modelo.
>
> Por otro lado, las categorías **Educación** y **Ocio y Servicios** obtuvieron los valores más bajos de F1-Score (0.59 y 0.64, respectivamente). Este comportamiento puede atribuirse a que algunas descripciones de transacciones contienen palabras o expresiones similares a las utilizadas en otras categorías, generando cierta ambigüedad durante la clasificación.
>
> Asimismo, la categoría **Transporte** alcanzó un Recall de **0.83**, indicando que el modelo identifica correctamente la mayoría de las transacciones pertenecientes a esta clase, aunque con una precisión ligeramente menor debido a la existencia de algunos falsos positivos.
>
> En términos generales, el modelo logró un desempeño consistente entre las diferentes categorías, alcanzando un **F1-Score macro de 0.71**, lo que indica un equilibrio adecuado en la capacidad de clasificación sin favorecer de manera significativa a alguna clase en particular.

- Se armo un Pipeline para poder guardar el modelo de `TfidfVectorizer`, esto hace que ahora entrene solo las palabras clave para predecir la categoría.


### Dataset Perfil Financiero (B)


In [7]:
features_perfil_financiero = [
    "ingreso_mensual",
    "nivel_endeudamiento",
    "ratio_gasto_ingreso",
    "frecuencia_ahorro",
]

X_rf = df_perfil_financiero[features_perfil_financiero]
y_rf = df_perfil_financiero["perfil_financiero"]

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y_rf, test_size=0.2, random_state=42
)

preprocesador = ColumnTransformer(
    transformers=[
        (
            "encoder_ahorro",
            OrdinalEncoder(categories=[["Baja", "Media", "Alta"]]),
            ["frecuencia_ahorro"],
        ),
    ],
    remainder="passthrough",
)

pipeline_perfil = Pipeline(
    [
        ("preprocesador", preprocesador),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

pipeline_perfil.fit(X_train_rf, y_train_rf)

output_dir = Path("../models")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "modelo_perfil_financiero.joblib"

joblib.dump(pipeline_perfil, output_path, compress=3)

# oci_service.subir_archivo(
#     nombre_objeto="modelo_perfil_financiero.joblib",  # Nombre en OCI (destino)
#     ruta_local=output_path                             # Ruta local (origen)
# )

['../models/modelo_perfil_financiero.joblib']

Se uso **Random Forest Classifier** según la interpretación de datos en _..\notebooks\03_luz_bi_analysis.ipynb_

> El modelo seleccionado obtuvo un **Accuracy del 93%** sobre el conjunto de prueba, lo que indica que fue capaz de clasificar correctamente la mayoría de los perfiles financieros simulados que no participaron durante el entrenamiento.
>
> Las métricas de **Precision**, **Recall** y **F1-Score** presentan valores cercanos o superiores al 0.90 para la mayoría de las clases, lo que refleja un desempeño consistente y un buen equilibrio entre la capacidad del modelo para identificar correctamente cada perfil financiero y minimizar las clasificaciones erróneas.
>
> En conjunto, los resultados muestran que el modelo posee una adecuada capacidad de generalización sobre los datos simulados y constituye una herramienta confiable para la clasificación automática del perfil financiero.

- Se utilizó un `ColumnTransformer` dentro de un `Pipeline` para inferencia en producción.


# Prueba de los modelos


In [8]:
"""Modelos en local"""
# modelo_descargado = oci_service.descargar_archivo("modelo_perfil_financiero.joblib")
# modelo_cargado = joblib.load("../models/descargados/modelo_perfil_financiero.joblib")
modelo_cargado = joblib.load("../models/modelo_perfil_financiero.joblib")

"""Modelo OCI en RAM"""
# modelo_cargado = oci_service.cargar_modelo_joblib(
#     "modelo_perfil_financiero.joblib"
# )

"""Modelo OCI en cache (carpeta temp)"""
# modelo_cargado = oci_service.cargar_modelo_con_cache(
#     "modelo_perfil_financiero.joblib"
# )

nuevo_usuario = pd.DataFrame(
    [
        {
            "ingreso_mensual": 25000,
            "nivel_endeudamiento": 0.3,
            "ratio_gasto_ingreso": 0.6,
            "frecuencia_ahorro": "Media",
        }
    ]
)

perfil = modelo_cargado.predict(nuevo_usuario)
print("El perfil del nuevo cliente es:", perfil[0])
perfil

El perfil del nuevo cliente es: Saludable


array(['Saludable'], dtype=object)

In [9]:
"""Modelos en local"""
# modelo_descargado_gastos = oci_service.descargar_archivo("clasificador_gastos.joblib")
# modelo_gastos = joblib.load("../models/descargados/clasificador_gastos.joblib")
modelo_gastos = joblib.load("../models/clasificador_gastos.joblib")

"""Modelo OCI en RAM"""
# modelo_gastos = oci_service.cargar_modelo_joblib(
#     "clasificador_gastos.joblib"
# )

"""Modelo OCI en cache (carpeta temp)"""
# modelo_gastos = oci_service.cargar_modelo_con_cache(
#     "clasificador_gastos.joblib"
# )


def preparar_gasto(descripcion):
    return [limpiar_texto(descripcion)]


# Prueba con cualquier texto libre
nuevo_gasto = preparar_gasto("pago de luz y agua")

categoria = modelo_gastos.predict(nuevo_gasto)
print("La categoría predicha es:", categoria[0])
categoria

La categoría predicha es: Vivienda


array(['Vivienda'], dtype=object)